In [ ]:
!pip install ultralytics opencv-python matplotlib pyyaml Pillow -q

import ultralytics
ultralytics.checks()

In [ ]:
import os

# For local execution, ensure the dataset zip is in the same directory as this notebook
ZIP_PATH = "Dataset_Helmet.zip"  

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"⚠️ {ZIP_PATH} not found! Please place the dataset zip file in the same directory as this notebook.")
else:
    print(f"✅ Found dataset zip: {os.path.abspath(ZIP_PATH)}")

In [ ]:
import zipfile

EXTRACT_DIR = './raw_dataset'

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Extracted contents:")
for d in os.listdir(EXTRACT_DIR):
    count = len(os.listdir(os.path.join(EXTRACT_DIR, d)))
    print(f"  {d}/ → {count} files")

In [ ]:
import xml.etree.ElementTree as ET
import shutil
import random
from pathlib import Path
from collections import Counter

# ── Configuration ──
IMAGES_DIR = os.path.join(EXTRACT_DIR, 'images')
ANNOTATIONS_DIR = os.path.join(EXTRACT_DIR, 'annotations')
DATASET_DIR = './helmet_dataset'
VAL_SPLIT = 0.2
RANDOM_SEED = 42

# Class mapping
CLASS_MAP = {
    'With Helmet': 0,
    'Without Helmet': 1,
}

def voc_to_yolo(xml_path, class_map):
    """Convert a Pascal VOC XML annotation to YOLO format."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    size = root.find('size')
    img_w = int(size.find('width').text)
    img_h = int(size.find('height').text)
    
    yolo_lines = []
    labels_found = []
    
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in class_map:
            print(f"  ⚠️ Unknown class '{name}' in {xml_path}, skipping")
            continue
            
        class_id = class_map[name]
        labels_found.append(name)
        
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        
        # Convert to YOLO format (normalized x_center, y_center, width, height)
        x_center = ((xmin + xmax) / 2.0) / img_w
        y_center = ((ymin + ymax) / 2.0) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        
        yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")
        
    return yolo_lines, labels_found


# ── Create directory structure ──
for split in ['train', 'val']:
    os.makedirs(os.path.join(DATASET_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR, split, 'labels'), exist_ok=True)

# ── Get all annotation files and shuffle ──
xml_files = sorted([f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.xml')])
random.seed(RANDOM_SEED)
random.shuffle(xml_files)

split_idx = int(len(xml_files) * (1 - VAL_SPLIT))
train_files = xml_files[:split_idx]
val_files = xml_files[split_idx:]

print(f"Total annotations: {len(xml_files)}")
print(f"Train set: {len(train_files)} images")
print(f"Val set:   {len(val_files)} images")
print()

# ── Convert and copy ──
all_labels = Counter()
skipped = 0

for split_name, file_list in [('train', train_files), ('val', val_files)]:
    for xml_file in file_list:
        xml_path = os.path.join(ANNOTATIONS_DIR, xml_file)
        stem = Path(xml_file).stem
        img_file = stem + '.png'
        img_path = os.path.join(IMAGES_DIR, img_file)
        
        if not os.path.exists(img_path):
            # Try jpg
            img_file = stem + '.jpg'
            img_path = os.path.join(IMAGES_DIR, img_file)
            if not os.path.exists(img_path):
                print(f"  ⚠️ Image not found for {xml_file}, skipping")
                skipped += 1
                continue
                
        yolo_lines, labels = voc_to_yolo(xml_path, CLASS_MAP)
        all_labels.update(labels)
        
        # Copy image
        shutil.copy2(img_path, os.path.join(DATASET_DIR, split_name, 'images', img_file))
        
        # Write YOLO label
        label_path = os.path.join(DATASET_DIR, split_name, 'labels', stem + '.txt')
        with open(label_path, 'w') as f:
            f.write('\n'.join(yolo_lines))

print(f"\n✅ Conversion complete!")
print(f"   Skipped: {skipped}")
print(f"\n📊 Label distribution:")
for label, count in all_labels.most_common():
    print(f"   {label}: {count}")

In [ ]:
# ── Verify the dataset structure ──
for split in ['train', 'val']:
    imgs = len(os.listdir(os.path.join(DATASET_DIR, split, 'images')))
    lbls = len(os.listdir(os.path.join(DATASET_DIR, split, 'labels')))
    print(f"{split}: {imgs} images, {lbls} labels")

# Show a sample label file
sample_label = os.listdir(os.path.join(DATASET_DIR, 'train', 'labels'))[0]
print(f"\nSample label ({sample_label}):")
with open(os.path.join(DATASET_DIR, 'train', 'labels', sample_label)) as f:
    print(f.read())

In [ ]:
import yaml

dataset_config = {
    'path': os.path.abspath(DATASET_DIR),
    'train': 'train/images',
    'val': 'val/images',
    'names': {
        0: 'With Helmet',
        1: 'Without Helmet',
    }
}

YAML_PATH = os.path.join(DATASET_DIR, 'dataset.yaml')
with open(YAML_PATH, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"✅ Dataset config saved to {YAML_PATH}")
print()
with open(YAML_PATH) as f:
    print(f.read())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

CLASS_NAMES = {0: 'With Helmet', 1: 'Without Helmet'}
COLORS = {0: '#00FF88', 1: '#FF4444'}  # Green for helmet, Red for no helmet

def plot_yolo_sample(img_path, label_path, ax):
    """Plot an image with YOLO bounding boxes."""
    img = Image.open(img_path)
    img_w, img_h = img.size
    ax.imshow(img)
    
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls_id = int(parts[0])
                xc, yc, w, h = map(float, parts[1:])
                
                # Convert back to pixel coords
                x1 = (xc - w/2) * img_w
                y1 = (yc - h/2) * img_h
                bw = w * img_w
                bh = h * img_h
                
                color = COLORS.get(cls_id, '#FFFFFF')
                rect = patches.Rectangle(
                    (x1, y1), bw, bh,
                    linewidth=2, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(
                    x1, y1 - 4, CLASS_NAMES.get(cls_id, '?'),
                    color='white', fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8)
                )
                
    ax.set_title(os.path.basename(img_path), fontsize=9)
    ax.axis('off')


# Plot 8 random samples from training set
train_imgs_dir = os.path.join(DATASET_DIR, 'train', 'images')
train_lbls_dir = os.path.join(DATASET_DIR, 'train', 'labels')
sample_imgs = random.sample(os.listdir(train_imgs_dir), min(8, len(os.listdir(train_imgs_dir))))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Training Samples with Annotations', fontsize=16, fontweight='bold')

for ax, img_file in zip(axes.flat, sample_imgs):
    img_path = os.path.join(train_imgs_dir, img_file)
    label_path = os.path.join(train_lbls_dir, Path(img_file).stem + '.txt')
    plot_yolo_sample(img_path, label_path, ax)

plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data=YAML_PATH,
    epochs=30,
    imgsz=640,
    batch=8,
    name='helmet_detection',
    patience=300,            # Early stopping patience
    save=True,
    save_period=10,         # Save checkpoint every 10 epochs
    plots=True,             # Generate training plots
    verbose=True,
)

In [ ]:
# ── Run validation on the best model ──
best_model_path = os.path.join('runs', 'detect', 'helmet_detection', 'weights', 'best.pt')

if not os.path.exists(best_model_path):
    # Find the latest run
    import glob
    candidates = sorted(glob.glob('runs/detect/helmet_detection*/weights/best.pt'))
    if candidates:
        best_model_path = candidates[-1]
        print(f"Using model: {best_model_path}")
    else:
        raise FileNotFoundError("No trained model found! Please run training first.")

model = YOLO(best_model_path)

# Validate
metrics = model.val(data=YAML_PATH)

print("\n" + "="*60)
print("📊 VALIDATION RESULTS")
print("="*60)
print(f"  mAP50:        {metrics.box.map50:.4f}")
print(f"  mAP50-95:     {metrics.box.map:.4f}")
print(f"  Precision:    {metrics.box.mp:.4f}")
print(f"  Recall:       {metrics.box.mr:.4f}")
print("="*60)

In [ ]:
# ── Display training plots ──
from IPython.display import Image as IPImage, display

run_dir = os.path.dirname(os.path.dirname(best_model_path))

plot_files = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'F1_curve.png',
    'PR_curve.png',
    'P_curve.png',
    'R_curve.png',
]

for pf in plot_files:
    plot_path = os.path.join(run_dir, pf)
    if os.path.exists(plot_path):
        print(f"\n{'─'*40}")
        print(f"📈 {pf}")
        print(f"{'─'*40}")
        display(IPImage(filename=plot_path, width=800))

In [ ]:
# ── Predict on validation images ──
val_imgs_dir = os.path.join(DATASET_DIR, 'val', 'images')
val_images = os.listdir(val_imgs_dir)
sample_val = random.sample(val_images, min(12, len(val_images)))
sample_paths = [os.path.join(val_imgs_dir, f) for f in sample_val]

# Run inference
results = model.predict(
    source=sample_paths,
    conf=0.25,
    save=True,
    save_txt=True,
    name='helmet_predictions',
)

In [ ]:
# ── Display predictions ──
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle('Model Predictions on Validation Images', fontsize=16, fontweight='bold')

for ax, result in zip(axes.flat, results):
    # Get the annotated image from results
    annotated = result.plot()  # Returns BGR numpy array
    annotated_rgb = annotated[:, :, ::-1]  # Convert BGR → RGB
    ax.imshow(annotated_rgb)
    
    # Count detections
    n_detections = len(result.boxes)
    ax.set_title(f"{os.path.basename(result.path)} ({n_detections} detections)", fontsize=9)
    ax.axis('off')

# Hide unused axes
for ax in axes.flat[len(results):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── Detailed detection summary ──
print("\n" + "="*70)
print("🔍 DETECTION SUMMARY")
print("="*70)

total_with = 0
total_without = 0

for result in results:
    fname = os.path.basename(result.path)
    boxes = result.boxes
    with_h = sum(1 for b in boxes if int(b.cls) == 0)
    without_h = sum(1 for b in boxes if int(b.cls) == 1)
    total_with += with_h
    total_without += without_h
    print(f"  {fname:30s} → With Helmet: {with_h}, Without Helmet: {without_h}")
    
print(f"\n  {'TOTAL':30s} → With Helmet: {total_with}, Without Helmet: {total_without}")
print("="*70)

In [ ]:
# ── Export to ONNX (widely supported for deployment) ──
model.export(format='onnx', imgsz=640)
print("\n✅ Model exported to ONNX format!")

In [ ]:
# ── Export to TorchScript ──
model.export(format='torchscript', imgsz=640)
print("\n✅ Model exported to TorchScript format!")

In [ ]:
# ── The model is saved locally! ──
print("📥 Model files are saved locally in the 'runs' directory.")

# Check best weights
if os.path.exists(best_model_path):
    print(f"  ✅ Best weights path: {os.path.abspath(best_model_path)}")
    
# Check ONNX model
onnx_path = best_model_path.replace('.pt', '.onnx')
if os.path.exists(onnx_path):
    print(f"  ✅ ONNX model path: {os.path.abspath(onnx_path)}")

print("\n🎉 Done! Your trained helmet detection model is ready for local deployment.")

In [ ]:
# Select a random validation image to test
import random
import matplotlib.pyplot as plt

test_dir = os.path.join(DATASET_DIR, 'val', 'images')
test_imgs = os.listdir(test_dir)
if test_imgs:
    test_img_path = os.path.join(test_dir, random.choice(test_imgs))
    print(f"Testing on local image: {test_img_path}")
    
    # Run prediction
    test_results = model.predict(
        source=test_img_path,
        conf=0.25,
        save=False,
    )
    
    # Display result
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    annotated = test_results[0].plot()
    ax.imshow(annotated[:, :, ::-1])
    ax.axis('off')
    
    n = len(test_results[0].boxes)
    ax.set_title(f'Predictions on {os.path.basename(test_img_path)} ({n} detections)', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Print details
    for box in test_results[0].boxes:
        cls_name = CLASS_NAMES.get(int(box.cls), 'Unknown')
        conf = float(box.conf)
        print(f"  → {cls_name}: {conf:.2%} confidence")
else:
    print("No test images found in validation directory.")